# Face Follower — Client (Laptop side)

Reads the MJPEG stream from the ESP32-CAM, detects a face with OpenCV's Haar Cascade (fast, no extra install — swap for MediaPipe or YOLOv8n later if you want more accuracy), decides a direction, and sends it to the ESP32's `/motor` endpoint.

**Before running:** update `ESP32_IP` below to match what the ESP32 printed in its Serial Monitor after connecting to WiFi.

**Fix in this version:** the cascade classifier was silently failing to load (OpenCV doesn't raise an error on a bad path — it just gives you an empty, unusable classifier), which caused a crash the first time it was used. It turns out some recent `opencv-contrib-python` wheels (yours included) simply don't ship the `cv2/data/*.xml` cascade files at all — that's a packaging gap in that release, not a bug in your setup. Cell 2 now: checks the default path, searches your installed `cv2` package as a fallback, and if the file truly isn't there, downloads it once from OpenCV's GitHub repo and caches it next to this notebook (so future runs work offline).

**If you run cell 1 (the pip uninstall/install), restart the kernel afterward before running anything else** — stale imports from before the package swap are the most common cause of the empty-cascade bug.

In [2]:
# If needed:
%pip uninstall -y opencv-python
%pip install opencv-contrib-python
import cv2
import requests
import time
import threading
print(cv2.__file__)

# NOTE: if you just ran the uninstall/install above for the first time in this
# kernel session, restart the kernel now (Kernel -> Restart) and re-run this
# cell before continuing. Otherwise cv2.data.haarcascades can point at a stale
# location left over from the previous install.

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\erinx\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
C:\Users\erinx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\cv2\__init__.py


In [3]:
# ==================== CONFIG ====================
ESP32_IP = "10.158.116.178"          # <-- change to your ESP32's actual IP
STREAM_URL = f"http://{ESP32_IP}:81/stream"
MOTOR_URL = f"http://{ESP32_IP}/motor"

# Decision thresholds (tune these after testing)
CENTER_DEADZONE = 0.15     # face center within +-15% of frame center = "forward", else turn
CLOSE_FACE_RATIO = 0.35    # if face width > 35% of frame width, robot is close enough -> stop
COMMAND_COOLDOWN = 0.4     # seconds between HTTP commands, avoid spamming the ESP32
# ==================================================

import os
import urllib.request

# Some recent opencv-python / opencv-contrib-python wheels (e.g. the 5.0.0.x
# builds) simply don't ship the cv2/data/*.xml cascade files in the wheel at
# all -- it's a known packaging gap, not something wrong with your setup. If
# that's the case here, we download the file once from OpenCV's GitHub repo
# and cache it next to this notebook so you only need internet the first time.
CASCADE_FILENAME = "haarcascade_frontalface_default.xml"
CASCADE_CACHE_PATH = os.path.join(os.getcwd(), CASCADE_FILENAME)
CASCADE_DOWNLOAD_URL = (
    "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/"
    + CASCADE_FILENAME
)

def load_face_cascade():
    """Load the Haar cascade robustly.

    cv2.CascadeClassifier(path) does NOT raise an error for a bad path -- it
    just returns an empty, unusable classifier, and the failure only shows up
    later as a cryptic 'Assertion failed: !empty()' when you call
    detectMultiScale(). So we check explicitly, try a few known locations,
    and if none of them work (e.g. this cv2 build ships no data folder at
    all) we download the file straight from OpenCV's GitHub repo and cache
    it locally.
    """
    # 1) Try the default packaged path.
    default_path = cv2.data.haarcascades + CASCADE_FILENAME
    print("Looking for cascade at:", default_path)
    clf = cv2.CascadeClassifier(default_path)
    if not clf.empty():
        print("Loaded cascade OK from the packaged cv2 data folder.")
        return clf

    # 2) Search the rest of the installed cv2 package, in case it's just in
    #    a different subfolder than cv2.data.haarcascades points to.
    print("Default path failed or file missing -- searching installed cv2 package...")
    cv2_dir = os.path.dirname(cv2.__file__)
    for root, _dirs, files in os.walk(cv2_dir):
        if CASCADE_FILENAME in files:
            found_path = os.path.join(root, CASCADE_FILENAME)
            print("Found at:", found_path)
            clf = cv2.CascadeClassifier(found_path)
            if not clf.empty():
                print("Loaded cascade OK from fallback path.")
                return clf

    # 3) Already downloaded a cached copy previously? Use it.
    if os.path.exists(CASCADE_CACHE_PATH):
        print("Trying previously cached copy at:", CASCADE_CACHE_PATH)
        clf = cv2.CascadeClassifier(CASCADE_CACHE_PATH)
        if not clf.empty():
            print("Loaded cascade OK from cache.")
            return clf

    # 4) Not found anywhere locally -- this cv2 build likely ships no data
    #    folder at all. Download it once from OpenCV's GitHub repo.
    print(f"Not found locally. Downloading from {CASCADE_DOWNLOAD_URL} ...")
    try:
        urllib.request.urlretrieve(CASCADE_DOWNLOAD_URL, CASCADE_CACHE_PATH)
    except Exception as e:
        raise RuntimeError(
            f"Could not find {CASCADE_FILENAME} locally and the download failed "
            f"({e}). Check your internet connection, or manually download it from "
            f"{CASCADE_DOWNLOAD_URL} and save it as {CASCADE_CACHE_PATH}."
        )

    clf = cv2.CascadeClassifier(CASCADE_CACHE_PATH)
    if clf.empty():
        raise RuntimeError(
            f"Downloaded {CASCADE_CACHE_PATH} but it still failed to load -- "
            "the download may be corrupted or blocked (e.g. by a proxy/firewall "
            "returning an HTML error page instead of the XML file). Try opening "
            "that file in a text editor to check it's real XML, or download it "
            f"manually from {CASCADE_DOWNLOAD_URL}."
        )
    print("Loaded cascade OK from freshly downloaded file. Cached at:", CASCADE_CACHE_PATH)
    return clf

face_cascade = load_face_cascade()
assert not face_cascade.empty(), "face_cascade is still empty -- see error above."

Looking for cascade at: C:\Users\erinx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\cv2\data\haarcascade_frontalface_default.xml
Default path failed or file missing -- searching installed cv2 package...
Trying previously cached copy at: c:\Users\erinx\Downloads\CODES\files\haarcascade_frontalface_default.xml
Loaded cascade OK from cache.


In [4]:
# Send motor commands on a background thread so a slow/failed request never
# blocks the video loop. Only the latest desired direction is kept.

_last_sent_dir = None
_last_sent_time = 0
_command_lock = threading.Lock()

def send_direction(direction: str):
    """Send a direction to the ESP32, but only if it changed or the cooldown passed."""
    global _last_sent_dir, _last_sent_time
    now = time.time()
    with _command_lock:
        if direction == _last_sent_dir and (now - _last_sent_time) < COMMAND_COOLDOWN:
            return
        _last_sent_dir = direction
        _last_sent_time = now

    def _worker():
        try:
            requests.get(MOTOR_URL, params={"dir": direction}, timeout=1.0)
        except requests.RequestException as e:
            print(f"[motor] request failed: {e}")

    threading.Thread(target=_worker, daemon=True).start()

In [5]:
def decide_direction(face_box, frame_w, frame_h):
    """face_box = (x, y, w, h) of the largest detected face, in pixels.
    Returns one of: forward, left, right, stop.
    """
    x, y, w, h = face_box
    face_center_x = x + w / 2
    frame_center_x = frame_w / 2
    offset = (face_center_x - frame_center_x) / frame_w  # -0.5 .. 0.5

    face_width_ratio = w / frame_w

    if face_width_ratio > CLOSE_FACE_RATIO:
        return "stop"  # close enough, don't bump into the person

    if offset < -CENTER_DEADZONE:
        return "left"
    elif offset > CENTER_DEADZONE:
        return "right"
    else:
        return "forward"

In [6]:
def run_face_follower(show_window=True, max_seconds=None):
    """Main loop: read stream, detect face, decide + send direction, show annotated frame.
    Press 'q' in the video window to stop (if show_window=True).
    Set max_seconds to auto-stop after N seconds (useful for running inside Jupyter
    without a window, e.g. on a headless machine).
    """
    if face_cascade.empty():
        print("face_cascade is empty -- re-run the cascade-loading cell before starting.")
        return

    cap = cv2.VideoCapture(STREAM_URL)
    if not cap.isOpened():
        print(f"Could not open stream at {STREAM_URL}. Check ESP32_IP and that the ESP32 is powered on.")
        return

    start_time = time.time()
    no_face_frames = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Frame read failed, retrying...")
                time.sleep(0.2)
                continue

            frame_h, frame_w = frame.shape[:2]
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(40, 40))

            if len(faces) > 0:
                no_face_frames = 0
                # Track the largest face (closest / most prominent one)
                largest = max(faces, key=lambda f: f[2] * f[3])
                x, y, w, h = largest
                direction = decide_direction(largest, frame_w, frame_h)
                send_direction(direction)

                if show_window:
                    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    cv2.putText(frame, direction, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX,
                                0.8, (0, 255, 0), 2)
            else:
                no_face_frames += 1
                # A few consecutive no-face frames before stopping, to avoid
                # jitter from single missed detections.2
                if no_face_frames > 5:
                    send_direction("stop")

            if show_window:
                cv2.imshow("Face Follower", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            if max_seconds and (time.time() - start_time) > max_seconds:
                break
    finally:
        send_direction("stop")
        cap.release()
        if show_window:
            cv2.destroyAllWindows()

In [10]:
# Run it. On some remote/Jupyter setups cv2.imshow won't have a display —
# in that case set show_window=False and max_seconds=30 to test blind for 30s.
run_face_follower(show_window=True)

[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Read timed out. (read timeout=1.0)
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Read timed out. (read timeout=1.0)
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Read timed out. (read timeout=1.0)
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Read timed out. (read timeout=1.0)
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=stop (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x27a76fb4090>, 'Connection to 10.158.116.178 timed out. (connect timeout=1.0)'))
[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=forward (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x27a7c20ef90>, 'Connection to 10.158.116.178 timed out. (con

KeyboardInterrupt: 

[motor] request failed: HTTPConnectionPool(host='10.158.116.178', port=80): Max retries exceeded with url: /motor?dir=stop (Caused by ConnectTimeoutError(<HTTPConnection(host='10.158.116.178', port=80) at 0x27a76fb4650>, 'Connection to 10.158.116.178 timed out. (connect timeout=1.0)'))


## Upgrade paths

- **More accurate detection:** swap the Haar Cascade block for MediaPipe:
  ```python
  import mediapipe as mp
  mp_face = mp.solutions.face_detection.FaceDetection(min_detection_confidence=0.5)
  # results = mp_face.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
  ```
  Still CPU-friendly and noticeably more robust to angle/lighting than Haar Cascade.

- **Actual YOLO:** once you want real YOLO, use `ultralytics`'s YOLOv8n (`pip install ultralytics`) with a face-detection weight or a person class, then extract the bounding box the same way `decide_direction()` expects `(x, y, w, h)` — the rest of this notebook doesn't need to change.

- **Smoother turning:** right now turns are full-speed pivots (ENA/ENB tied to 5V). If you later free up 2 more GPIOs, wiring ENA/ENB to the ESP32 with `analogWrite`/`ledc` PWM would let you do proportional turning instead of full pivots.